In [ ]:
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
import warnings
from pint.errors import UnitStrippedWarning
warnings.simplefilter("ignore", UnitStrippedWarning)
import pandas as pd
import xarray as xr

import prism
import pym

from imagematerials.model import (
    GenericStocks,
    MaterialIntensities,
    SharesInflowStocks,
    ElectricVehicleBatteries,
    EvBatteryLinkModule
)
from imagematerials.factory import ModelFactory
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.concepts import create_h2_graph, create_materials_graph, KnowledgeGraph, Node
from imagematerials.util import resolve_circular_economy_scenario
from imagematerials.read_mym import read_mym_df, mymarray_to_xarray

from imagematerials.electricity.constants import STANDARD_SCEN_EXTERNAL_DATA, EPG_TECHNOLOGIES_FINAL
from imagematerials.electricity.utils import rebroadcast_type_dims,interpolate_xr
from imagematerials.electricity.constants import (
    STD_LIFETIMES_ELECTR,
    YEAR_FIRST_GRID
)
from imagematerials.hydrogen.constants import (
    YEAR_FIRST_H2_PRODUCTION,
    ELECTROLYSIS_TECHNOLOGIES
)

# TODO absolute path of file "preprocessing.py" ? current solution can differ depending on IDE used
path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials
path_base = Path(path_base, "data", "raw")
ce_scenario_path = Path(path_base, "circular_economy_scenarios")

graph_h2 = create_h2_graph()

year_start = YEAR_FIRST_GRID
year_end = 2100

COLOURS_HYDROGEN = {
    'ClGs':   '#1B9E77',
    'POOl':   '#D95F02',
    'SMR':    '#7570B3',
    'BioGs':  '#66A61E',
    'ClGsCS': '#66C2A5',
    'POOlCS': '#FC8D62',
    'SMRCS':  '#8DA0CB',
    'BioGsCS':'#A6D854',
    'BLEl':   '#E6AB02',
    'CurEl':  '#E7298A',
    'SPVEl':  '#A6761D',
    'CSPEl':  '#1F78B4',
    'WONEl':  '#B2DF8A',
    'WOFFEl': '#FDBF6F',
    'GEOEl':  '#CAB2D6',
    'ssSMR':  '#6A3D9A',
    'ssEl':   '#B15928',
}

HYDROGEN_TECHNOLOGIES = [
    'ClGs',
    'POOl',
    'SMR',
    'BioGs',
    'ClGsCS',
    'POOlCS',
    'SMRCS',
    'BioGsCS',
    'BLEl',
    'CurEl',
    'SPVEl',
    'CSPEl',
    'WONEl',
    'WOFFEl',
    'GEOEl',
    'ssSMR',
    'ssEl']

ELECTROLYSIS_TECHNOLOGIES = [
    'BLEl',
    'CurEl',
    'SPVEl',
    'CSPEl',
    'WONEl',
    'WOFFEl',
    'GEOEl',
    'ssEl'
]

# model run

In [ ]:
scenario_list = {
                # "SSP2_baseline_base":    ("SSP2_baseline", "base"),
                # "SSP2_baseline_full_ce": ("SSP2_baseline", "full_ce"),
                "SSP2_baseline":    ("SSP2_baseline", None),
                "SSP2_mitigation":   ("SSP2_mitigation", None),
                }

time_start = 2000
time_end = 2060
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
    circular_economy_scenario_dir = resolve_circular_economy_scenario(ce_scenario_path, circular_scen)

    h2_sector = get_preprocessing_data("hydrogen", path_base,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False,
                                    resource_efficiency_flags_file=circular_economy_scenario_dir) 
    
    factory = ModelFactory(
    h2_sector, complete_timeline
    ).add(SharesInflowStocks, ["hydrogen"],
    ).add(MaterialIntensities, ["hydrogen"],
    )
    model = factory.finish()
    
    model.simulate(simulation_timeline)

    all_output[scen_id] = model

    print(f"Finished {scen_id}")
    # list(model.elc_stor_other)


# TIMER Data

In [ ]:
h2cap = mymarray_to_xarray(pym.read_mym(Path(path_base, "image", "SSP2_baseline", "EnergyServices", "H2Cap.out")),
                          dim_1="NRT", dim_2="HydrogenSourcesTotal")
h2prod = mymarray_to_xarray(pym.read_mym(Path(path_base, "image", "SSP2_baseline", "EnergyServices", "H2ProdSpec.out")),
                          dim_1="NRT", dim_2="HydrogenSources")
h2cap = prism.Q_(h2cap, "GW")
h2prod = prism.Q_(h2prod, "GJ")

h2cap = h2cap.rename({'t': 'time','NRT': 'Region','HydrogenSourcesTotal': 'Type'})
h2prod = h2prod.rename({'t': 'time','NRT': 'Region','HydrogenSources': 'Type'})

h2cap = h2cap.drop_sel(Type='total')

def get_h2cap(path, scen):
    h2cap = mymarray_to_xarray(pym.read_mym(Path(path, "image", scen, "EnergyServices", "H2Cap.out")),
                          dim_1="NRT", dim_2="HydrogenSourcesTotal")
    h2cap = prism.Q_(h2cap, "GW")
    h2cap = h2cap.rename({'t': 'time','NRT': 'Region','HydrogenSourcesTotal': 'Type'})
    h2cap = h2cap.drop_sel(Type='total')

    return h2cap

h2cap_b = get_h2cap(path_base, "SSP2_baseline")
h2cap_m = get_h2cap(path_base, "SSP2_mitigation")


# Other Data

In [ ]:
# material compositions of hydrogen production technologies (kg/MW)
material_intensities = pd.read_csv(path_base / "hydrogen" / "standard_data" / "hydrogen_material_intensities_long.csv",
                        usecols=["time","technology","sub_technology","material","value"])

# market shares
market_shares = pd.read_csv(path_base / "hydrogen" / "standard_data" / "hydrogen_market_shares_long.csv",
                        usecols=["time","technology","sub_technology","value"])

# material intensities (kg/MW) -----------------------------------------------------------------
# create new column with sub-technology name if available, otherwise technology name
# material_intensities["Type"] = material_intensities["sub_technology"].fillna(material_intensities["technology"])
material_intensities["Type"] = (
        material_intensities["technology"] + "_" + material_intensities["sub_technology"]
    )
material_intensities_da = (
    material_intensities.set_index(["time", "Type", "material"])["value"]
    .to_xarray()
    .rename({"time": "Cohort"})
)
material_intensities_da = prism.Q_(material_intensities_da, "kg/MW")
material_intensities_da.name = "HydrogenMaterialIntensities"
# material_intensities_da = knowledge_graph_electr.rebroadcast_xarray(material_intensities_da, output_coords=EPG_SUB_TECHNOLOGIES, dim="Type")

# market shares (%) ----------------------------------------------------------------------------
market_shares["Type"] = (
            market_shares["technology"] + "_" + market_shares["sub_technology"]
        )
market_shares_da = (
    market_shares.set_index(["time", "Type"])["value"]
    .to_xarray()
    .rename({"time": "Cohort"})
)
market_shares_da = prism.Q_(market_shares_da, "dimensionless")
market_shares_da.name = "HydrogenMarketShares"
market_shares_da_interp = interpolate_xr(market_shares_da, year_start, year_end)

## market shares interpolation

In [ ]:
# timer
# choose scen here:
h2cap_alltypes = h2cap_b

h2cap_electrolysis = h2cap_alltypes.sel(Type=ELECTROLYSIS_TECHNOLOGIES)
types = ["IMAGE-energy PEMEL", "IMAGE-energy AEL/SOEL"]
test = graph_h2.aggregate_sum(h2cap_electrolysis, output_coords=types).sum(dim="Region")
ms = test / test.sum(dim="Type")
ms = ms.rename({"time": "Cohort"})
mapping = {
    "IMAGE-energy PEMEL": "PEMEL",
    "IMAGE-energy AEL/SOEL": "AEL/SOEL",
}
ms = ms.assign_coords(Type=[mapping.get(t, t) for t in ms["Type"].values])

# cml
# 1. 
shares_el = market_shares_da_interp.sel(Type=["electrolysis_AEL", "electrolysis_PEMEL", "electrolysis_SOEL"])
# 2.
shares_el_agg = graph_h2.aggregate_sum(shares_el, output_coords=["PEMEL", "AEL/SOEL"])
# 3.
shares_ael_soel = shares_el.sel(Type = ["electrolysis_AEL", "electrolysis_SOEL"])
mapping = {
    "electrolysis_AEL": "AEL",
    "electrolysis_SOEL": "SOEL",
}
shares_ael_soel = shares_ael_soel.assign_coords(Type=[mapping.get(t, t) for t in shares_ael_soel["Type"].values])
# shares_ael_soel = shares_ael_soel.rename({"electrolysis_AEL": "AEL", "electrolysis_SOEL": "SOEL"})
shares_ael_soel = shares_ael_soel / shares_ael_soel.sum(dim="Type")

# 4.-6
x2 = xr.concat([shares_el_agg.sel(Cohort=slice(None,2030)), ms.sel(Cohort=slice(2040,None))], dim="Cohort")
x2_interp = interpolate_xr(x2, year_start, year_end)

# 7.
x3 = x2_interp.copy()
ael = (x3.sel(Type="AEL/SOEL", drop=True) * shares_ael_soel.sel(Type="AEL", drop=True)).expand_dims(Type=["AEL"])
soel = (x3.sel(Type="AEL/SOEL", drop=True)  * shares_ael_soel.sel(Type="SOEL", drop=True)).expand_dims(Type=["SOEL"]) 
x3 = xr.concat([x3, ael, soel], dim="Type")

# 8.
shares_final = market_shares_da_interp.copy()
mapping = {
    "electrolysis_AEL": "AEL",
    "electrolysis_SOEL": "SOEL",
    "electrolysis_PEMEL": "PEMEL",
}

cohorts = shares_final["Cohort"].sel(Cohort=slice(2031, None)).values  # "after 2030"; use 2030 if you want it included

for target, source in mapping.items():
    shares_final.loc[dict(Type=target, Cohort=cohorts)] = (
        x3.sel(Type=source, Cohort=cohorts).values
    )

# Plots

## model OUTPUT

In [ ]:

# %% Config -----------------------------------------------------------------
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
 
# harmless warning when .values strips the pint units for matplotlib
warnings.filterwarnings("ignore", message="The unit of the quantity is stripped")
 
SCENARIOS = {                          # label -> key in all_output
    "Baseline": "SSP2_baseline",
    "Mitigation": "SSP2_mitigation",
}
T0, T1 = 2015, 2060                    # plotting window
CUM_T0 = 2020                          # start year of the cumulative sums (fig 6)
N_REGIONS = 8                          # regions shown individually in fig 3 (max 10)
OUTDIR = "figures"
 
COLS = ["Inflow", "Stock", "Outflow"]
CAP_UNIT = {"Inflow": "GW/yr", "Stock": "GW", "Outflow": "GW/yr"}   # assumes annual time steps
 
# "bulk" materials
BULK = ["concrete", "steel", "aluminium", "copper"]
 
TYPE_COLORS = {
    "CBF_tech1": "#6b6b6b",
    "CBF+CCS_tech1": "#b08968",
    "electrolysis_AEL": "#1f77b4",
    "electrolysis_PEMEL": "#2ca02c",
    "electrolysis_SOEL": "#ff7f0e",
}

# Process data ----------------------------------------------------------------
# Pull the six arrays per scenario out of all_output into two nested dicts:
#   cap[scenario][column] -> capacity in GW   (dims: time, [cohort,] Region, Type)
#   mat[scenario][column] -> materials in kg  (dims: time, [cohort,] Region, Type, material)
cap, mat = {}, {}
for scen, key in SCENARIOS.items():
    h = all_output[key].hydrogen
    cap[scen] = {"Inflow": h["inflow"].to_array(),
                 "Stock": h["stock_by_cohort"],
                 "Outflow": h["outflow_by_cohort"].to_array()}
    mat[scen] = {"Inflow": h["inflow_materials"].to_array(),
                 "Stock": h["stock_by_cohort_materials"],
                 "Outflow": h["outflow_by_cohort_materials"].to_array()}
 
# Sum out any extra dimension (e.g. a cohort axis) and cut to the plotting window.
# The material arrays run 1926-2100, the capacity arrays 2000-2060.
for group in (cap, mat):
    for scen in group:
        for col, da in group[scen].items():
            extra = [d for d in da.dims if d not in ("time", "Region", "Type", "material")]
            if extra:
                da = da.sum(extra)
            group[scen][col] = da.sel(time=slice(T0, T1))
 
base, mit = list(SCENARIOS)[:2]
MATERIALS = [str(m) for m in mat[base]["Inflow"].material.values]
MINOR = [m for m in MATERIALS if m not in BULK]
MAT_COLORS = dict(zip(MATERIALS, plt.get_cmap("tab20").colors))   # same colour per material in all figs
 
# quick sanity check: dims and final-year totals
for name, group in (("capacity", cap), ("materials", mat)):
    for scen in group:
        for col in COLS:
            da = group[scen][col]
            print(f"{name:9s} {scen:10s} {col:8s} dims={da.dims} "
                  f"time={int(da.time[0])}-{int(da.time[-1])} "
                  f"final-year total={float(da.sum(['Region', 'Type'] + (['material'] if 'material' in da.dims else [])).isel(time=-1)):.4g}")



### Fig1: capacity per tech.

In [ ]:
# %% Fig 1: global totals (both scenarios) + capacity by technology per scenario
fig, axes = plt.subplots(1 + len(cap), 3, figsize=(15, 3.3 * (1 + len(cap)) + 0.8),
                         sharex=True, constrained_layout=True)
scen_colors = {base: "#444444", mit: "#d62728"}

# row 0: global total of both scenarios in the same panel
for j, col in enumerate(COLS):
    for s in (base, mit):
        tot = cap[s][col].sum(["Region", "Type"])                      # sum over region and technology
        axes[0, j].plot(tot.time.values, tot.values, label=s, color=scen_colors[s], lw=2)
    axes[0, j].set_title(f"{col} [{CAP_UNIT[col]}]")
axes[0, 0].set_ylabel("Global total", fontweight="bold")
axes[0, -1].legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)

# rows 1...: one row per scenario, stacked by technology
for i, scen in enumerate(cap, start=1):
    for j, col in enumerate(COLS):
        g = cap[scen][col].sum("Region").transpose("Type", "time")      # sum over region -> (Type, time)
        types = [str(t) for t in g.Type.values]
        axes[i, j].stackplot(g.time.values, g.values, labels=types,
                             colors=[TYPE_COLORS[t] for t in types])
    axes[i, 0].set_ylabel(scen, fontweight="bold")
h_, l_ = axes[1, -1].get_legend_handles_labels()
axes[1, -1].legend(h_[::-1], l_[::-1], loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)

# axis limits AFTER all rows are plotted (so the shared y-axis covers every row of a column)
for ax in axes.flat:
    ax.set_xlim(T0, T1)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)
fig.suptitle("Hydrogen production capacity (global): totals and technology mix", fontweight="bold")
# fig.savefig(f"{OUTDIR}/fig1_capacity.png", dpi=300, bbox_inches="tight")
plt.show()

### Fig2: capacity per region

In [ ]:
# %% Fig 2: capacity by region (top N regions + rest) ------------------------
# rank regions by their peak stock (summed over technologies and both scenarios)
score = sum(cap[s]["Stock"].sum("Type").max("time") for s in cap)
top = [str(r) for r in score.sortby(score, ascending=False).Region.values[:N_REGIONS]]
region_colors = list(plt.get_cmap("tab10").colors[:N_REGIONS]) + ["lightgrey"]
 
fig, axes = plt.subplots(len(cap), 3, figsize=(15, 7), sharex=True,# sharey="col",
                         constrained_layout=True)
for i, scen in enumerate(cap):
    for j, col in enumerate(COLS):
        da = cap[scen][col].sum("Type")                                   # (time, Region)
        top_vals = da.sel(Region=top).transpose("Region", "time").values
        rest = da.drop_sel(Region=top).sum("Region").values               # all other regions lumped
        ax = axes[i, j]
        ax.stackplot(da.time.values, np.vstack([top_vals, rest]),
                     labels=top + ["Rest"], colors=region_colors)
        ax.set_xlim(T0, T1)
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.3)
    axes[i, 0].set_ylabel(scen, fontweight="bold")
for j, col in enumerate(COLS):
    axes[0, j].set_title(f"{col} [{CAP_UNIT[col]}]")
h_, l_ = axes[0, -1].get_legend_handles_labels()
axes[0, -1].legend(h_[::-1], l_[::-1], loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
fig.suptitle(f"Hydrogen production capacity by region (top {N_REGIONS} + rest)", fontweight="bold")
# fig.savefig(f"{OUTDIR}/fig3_capacity_by_region.png", dpi=300, bbox_inches="tight")
plt.show()


### Fig3: bulk materials

In [ ]:

# %% Fig 3: bulk materials (stacked, Mt) -------------------------------------
fig, axes = plt.subplots(len(mat), 3, figsize=(15, 7), sharex=True,# sharey="col",
                         constrained_layout=True)
for i, scen in enumerate(mat):
    for j, col in enumerate(COLS):
        g = mat[scen][col].sel(material=BULK).sum(["Region", "Type"]).transpose("material", "time")
        ax = axes[i, j]
        ax.stackplot(g.time.values, g.values * 1e-9, labels=BULK,      # kg -> Mt
                     colors=[MAT_COLORS[m] for m in BULK])
        ax.set_xlim(T0, T1)
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.3)
    axes[i, 0].set_ylabel(scen, fontweight="bold")
for j, col in enumerate(COLS):
    axes[0, j].set_title(f"{col} [{'Mt' if col == 'Stock' else 'Mt/yr'}]")
h_, l_ = axes[0, -1].get_legend_handles_labels()
axes[0, -1].legend(h_[::-1], l_[::-1], loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
fig.suptitle("Bulk materials in hydrogen production capacity (global)", fontweight="bold")
# fig.savefig(f"{OUTDIR}/fig4_bulk_materials.png", dpi=300, bbox_inches="tight")
plt.show()

### Fig 4: CMs materials

In [ ]:

# %% Fig 5: minor / critical materials (lines, log scale, kt) ----------------
fig, axes = plt.subplots(len(mat), 3, figsize=(15, 7), sharex=True,# sharey="col",
                         constrained_layout=True)
for i, scen in enumerate(mat):
    for j, col in enumerate(COLS):
        g = mat[scen][col].sel(material=MINOR).sum(["Region", "Type"]).transpose("material", "time")
        ax = axes[i, j]
        for k, m in enumerate(MINOR):
            y = g.values[k] * 1e-6                                       # kg -> kt
            ax.plot(g.time.values, np.where(y > 0, y, np.nan), label=m,  # zeros -> NaN for log axis
                    color=MAT_COLORS[m], lw=1.6)
        ax.set_yscale("log")
        ax.set_xlim(T0, T1)
        ax.grid(alpha=0.3, which="both")
    axes[i, 0].set_ylabel(scen, fontweight="bold")
for j, col in enumerate(COLS):
    axes[0, j].set_title(f"{col} [{'kt' if col == 'Stock' else 'kt/yr'}]")
axes[0, -1].legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
fig.suptitle("Minor / critical materials (global, log scale)", fontweight="bold")
# fig.savefig(f"{OUTDIR}/fig5_minor_materials.png", dpi=300, bbox_inches="tight")
plt.show()


### Fig5: heatmap-cumulative material inflow

In [ ]:

# %% Fig 5: cumulative material inflow by technology (heatmaps) + % change ---
# cumulative inflow CUM_T0-T1, summed over regions -> (material, Type)
cum = {s: mat[s]["Inflow"].sel(time=slice(CUM_T0, T1)).sum(["time", "Region"])
          .transpose("material", "Type") for s in (base, mit)}
types = [str(t) for t in cum[base].Type.values]
order = np.argsort(-cum[base].sum("Type").values)                       # largest baseline demand first
materials = [str(cum[base].material.values[k]) for k in order]
cum = {s: cum[s].values[order] * 1e-6 for s in cum}                    # numpy arrays, kg -> kt
 
allvals = np.concatenate([v.ravel() for v in cum.values()])
norm = LogNorm(allvals[allvals > 0].min(), allvals.max())              # same colour scale for both heatmaps
 
fig, axes = plt.subplots(1, 3, figsize=(19, 6.2), constrained_layout=True,
                         gridspec_kw={"width_ratios": [1, 1, 0.8]})
for ax, s in zip(axes[:2], (base, mit)):
    im = ax.imshow(np.ma.masked_less_equal(cum[s], 0), norm=norm, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(types)))
    ax.set_xticklabels(types, rotation=40, ha="right")
    ax.set_yticks(range(len(materials)))
    ax.set_yticklabels(materials if s == base else [])
    for r in range(len(materials)):
        for c in range(len(types)):
            v = cum[s][r, c]
            if v > 0:
                ax.text(c, r, f"{v:.2g}", ha="center", va="center", fontsize=8,
                        color="w" if norm(v) < 0.55 else "k")
    ax.set_title(f"{s}: cumulative inflow {CUM_T0}-{T1} [kt]")
fig.colorbar(im, ax=axes[:2], shrink=0.8, label="kt (log scale)")
 
tot_b, tot_m = cum[base].sum(axis=1), cum[mit].sum(axis=1)             # total over technologies
pct = np.where(tot_b > 0, (tot_m / np.where(tot_b > 0, tot_b, 1) - 1) * 100, np.nan)
axes[2].barh(range(len(materials)), pct, color=np.where(pct < 0, "tab:blue", "tab:orange"))
axes[2].set_ylim(len(materials) - 0.5, -0.5)                           # same row order as heatmaps
axes[2].set_yticks(range(len(materials)))
axes[2].set_yticklabels([])
axes[2].axvline(0, color="k", lw=0.8)
axes[2].set_xlabel("%")
axes[2].set_title(f"Change in total cumulative inflow\n({mit} vs {base})")
axes[2].grid(alpha=0.3, axis="x")
# fig.savefig(f"{OUTDIR}/fig6_material_by_technology.png", dpi=300, bbox_inches="tight")
plt.show()


## INPUT data

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for t in h2cap["Type"].values:
    h2cap.sel(Type=t, Region="WORLD", time=slice(2010,2070)).plot.line(x="time", ax=ax, color=COLOURS_HYDROGEN[t], label=str(t))

ax.legend()


In [ ]:
["FF (mainly SMR)", "biofuel", "gasification + CCS", "electrolysis"]
["carbon-based fuel (mainly SMR)", "carbon-based fuel + CCS", "electrolysis"]
["carbon-based fuel (mainly SMR)", "carbon-based fuel + CCS", "ElAEL"]
types = ["coal", "oil", "natural_gas", "biofuel", "electrolysis"]
test = graph_h2.aggregate_sum(h2cap, output_coords=types)

fig, ax = plt.subplots(figsize=(10, 6))
for t in test["Type"].values:
    test.sel(Type=t, Region="WORLD", time=slice(2010,2070)).plot.line(x="time", ax=ax, label=str(t))

ax.legend()

In [ ]:
h2cap_b

In [ ]:
# 1 SUBPLOT

types = ["CBF", "CBF+CCS", "electrolysis"]
# types = ["coal", "oil", "natural_gas", "Bio", "electrolysis"]

var1 = graph_h2.aggregate_sum(h2cap_b, output_coords=types)
var2 = graph_h2.aggregate_sum(h2cap_m, output_coords=types)

fig, ax = plt.subplots(figsize=(10, 6))
for t in var1["Type"].values:
    var1.sel(Type=t, Region="WORLD", time=slice(2010,2070)).plot.line(x="time", ax=ax, label=str(t))
    var2.sel(Type=t, Region="WORLD", time=slice(2010,2070)).plot.line(x="time", ax=ax, label=str(t), linestyle=":")

ax.legend()

In [ ]:
# 2 SUBPLOTS #

types = ["CBF", "CBF+CCS", "electrolysis"]
# types = ["coal", "oil", "natural_gas", "Bio", "electrolysis"]

var1 = graph_h2.aggregate_sum(h2cap_b, output_coords=types)
var2 = graph_h2.aggregate_sum(h2cap_m, output_coords=types)

fig, (ax1, ax2) = plt.subplots(2,1, figsize=(10, 8))
fig.subplots_adjust(hspace=0.3)
for t in var1["Type"].values:
    var1.sel(Type=t, Region="WORLD", time=slice(2010,2070)).plot.line(x="time", ax=ax1, label=str(t), linewidth=2) #, linestyle=":"
    var2.sel(Type=t, Region="WORLD", time=slice(2010,2070)).plot.line(x="time", ax=ax2, label=str(t), linewidth=2) #, linestyle=":"

ax1.ticklabel_format(axis="y", style="sci", scilimits=(5, 5), useMathText=True)
ax2.ticklabel_format(axis="y", style="sci", scilimits=(5, 5), useMathText=True)
ax1.yaxis.get_offset_text().set_fontsize(14)
ax2.yaxis.get_offset_text().set_fontsize(14)

ax1.tick_params(axis="both", labelsize=14)
ax2.tick_params(axis="both", labelsize=14)
ax1.set_ylabel("H2 Prod. Capacity (GW)", fontsize=14)
ax2.set_ylabel("H2 Prod. Capacity (GW)", fontsize=14)
ax1.legend(fontsize=12)
ax1.set_title("SSP2 baseline")
ax2.legend(fontsize=12)
ax2.set_title("SSP2 VLLO")

### market shares TIMER

In [ ]:
# 1 SUBPLOT

select = ["electrolysis+RE", "BLEl", "CurEl", "ssEl"]

var_el = h2cap_b.sel(Type=ELECTROLYSIS_TECHNOLOGIES)
# var_el = h2cap_m.sel(Type=ELECTROLYSIS_TECHNOLOGIES)
var1 = graph_h2.aggregate_sum(var_el, output_coords=select)

fig, ax = plt.subplots(figsize=(10, 6))
for t in var1["Type"].values:
    var1.sel(Type=t, Region="WORLD", time=slice(2010,2070)).plot.line(x="time", ax=ax, label=str(t))
    # var2.sel(Type=t, Region="WORLD", time=slice(2010,2070)).plot.line(x="time", ax=ax, label=str(t), linestyle=":")

ax.legend()

In [ ]:
types = ["electrolysis+RE", "BLEl", "CurEl", "ssEl"]
COLORS_ELECTROLYSIS = {
    "electrolysis+RE": "#0224E6",
    "BLEl": "#F7B603",
    "CurEl": "#5998D6",
    "ssEl": "#E95E0D"
}

var_el = h2cap_b.sel(Type=ELECTROLYSIS_TECHNOLOGIES)
var1 = graph_h2.aggregate_sum(var_el, output_coords=types)
var_el = h2cap_m.sel(Type=ELECTROLYSIS_TECHNOLOGIES)
var2 = graph_h2.aggregate_sum(var_el, output_coords=types)

fig, (ax1, ax2) = plt.subplots(2,1, figsize=(10, 8))
fig.subplots_adjust(hspace=0.3)
for t in var1["Type"].values:
    var1.sel(Type=t, Region="WORLD", time=slice(2010,2070)).plot.line(x="time", ax=ax1, label=str(t), linewidth=2.5, color=COLORS_ELECTROLYSIS.get(t)) #, linestyle=":"
    var2.sel(Type=t, Region="WORLD", time=slice(2010,2070)).plot.line(x="time", ax=ax2, label=str(t), linewidth=2.5, color=COLORS_ELECTROLYSIS.get(t)) #, linestyle=":"

ax1.ticklabel_format(axis="y", style="sci", scilimits=(5, 5), useMathText=True)
ax2.ticklabel_format(axis="y", style="sci", scilimits=(5, 5), useMathText=True)
ax1.yaxis.get_offset_text().set_fontsize(14)
ax2.yaxis.get_offset_text().set_fontsize(14)

ax1.tick_params(axis="both", labelsize=14)
ax2.tick_params(axis="both", labelsize=14)
ax1.set_ylabel("H2 Prod. Capacity (GW)", fontsize=14)
ax2.set_ylabel("H2 Prod. Capacity (GW)", fontsize=14)
ax1.legend(fontsize=12)
ax1.set_title("electrolysis only - SSP2 baseline")
ax2.legend(fontsize=12)
ax2.set_title("electrolysis only - SSP2 VLLO")

### TIMER-CML

In [ ]:
for t in h2cap_b["time"].values:
    if total.sel(Region="WORLD", time=t)>0:
        first_year = t
        break
first_year

In [ ]:
# TIMER
select = ["electrolysis+RE", "BLEl", "CurEl", "ssEl"]

var_el = h2cap_b.sel(Type=ELECTROLYSIS_TECHNOLOGIES)
# var_el = h2cap_m.sel(Type=ELECTROLYSIS_TECHNOLOGIES)

var1 = graph_h2.aggregate_sum(var_el, output_coords=select)
total = var1.sum(dim="Type")
timer_pem = var1.sel(Type=["electrolysis+RE", "CurEl"]).sum(dim="Type")
timer_other = var1.sel(Type=["BLEl", "ssEl"]).sum(dim="Type")
timer_pem_shares = (timer_pem / total).sel(Region="WORLD", time=slice(2005,2070))
timer_other_shares = (timer_other / total).sel(Region="WORLD", time=slice(2005,2070))

# Liang
liang_pem_shares = market_shares_da.sel(Type=["electrolysis_PEMEL"]).sum(dim="Type")
liang_other_shares = market_shares_da.sel(Type=["electrolysis_AEL", "electrolysis_SOEL"]).sum(dim="Type")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(liang_pem_shares.Cohort, liang_pem_shares.values, color="#315370", linestyle =":", linewidth=2, label="CML data: PEM")
ax.plot(liang_other_shares.Cohort, liang_other_shares.values, color = "#E95E0D", linestyle =":", linewidth=2, label="CML data: AEL/SOEC")
ax.plot(timer_pem_shares.time, timer_pem_shares.values, color="blue", linestyle="-", linewidth=2, label="IMAGE-energy: PEM")
ax.plot(timer_other_shares.time, timer_other_shares.values, color="orange", linestyle="-", linewidth=2, label="IMAGE-energy: AEL/SOEC")

ax.set_title(r"H2 market shares - $\mathbf{SSP2\ baseline}$")
# ax.set_title(r"H2 market shares - $\mathbf{SSP2\ VLLO}$")

ax.tick_params(axis="both", labelsize=14)

ax.legend(fontsize=12, loc="upper right")


### TIMER-CML-Merged

In [ ]:
# TIMER
select = ["electrolysis+RE", "BLEl", "CurEl", "ssEl"]

var_el = h2cap_b.sel(Type=ELECTROLYSIS_TECHNOLOGIES)
# var_el = h2cap_m.sel(Type=ELECTROLYSIS_TECHNOLOGIES)

var1 = graph_h2.aggregate_sum(var_el, output_coords=select)
total = var1.sum(dim="Type")
timer_pem = var1.sel(Type=["electrolysis+RE", "CurEl"]).sum(dim="Type")
timer_other = var1.sel(Type=["BLEl", "ssEl"]).sum(dim="Type")
timer_pem_shares = (timer_pem / total).sel(Region="WORLD", time=slice(2005,2070))
timer_other_shares = (timer_other / total).sel(Region="WORLD", time=slice(2005,2070))
final_plot = shares_final.sel(Cohort=slice(2005,2070))


# Liang
liang_pem_shares = market_shares_da.sel(Type=["electrolysis_PEMEL"]).sum(dim="Type")
liang_other_shares = market_shares_da.sel(Type=["electrolysis_AEL", "electrolysis_SOEL"]).sum(dim="Type")

fig, ax = plt.subplots(figsize=(10, 6))
lw = 3
ax.plot(liang_pem_shares.Cohort, liang_pem_shares.values, color="#315370", linewidth=lw, label="CML data: PEM") #, linestyle =":"
ax.plot(liang_other_shares.Cohort, liang_other_shares.values, color = "#E95E0D", linewidth=lw, label="CML data: AEL/SOEL")
ax.plot(timer_pem_shares.time, timer_pem_shares.values, color="blue", linestyle="-", linewidth=lw, label="IMAGE-energy: PEM")
ax.plot(timer_other_shares.time, timer_other_shares.values, color="orange", linestyle="-", linewidth=lw, label="IMAGE-energy: AEL/SOEL")
ax.plot(final_plot.Cohort, final_plot.sel(Type="electrolysis_PEMEL").values, color="#6F9DFF", linestyle ="--", linewidth=lw+0.5, label="MERGED: PEM")
ax.plot(final_plot.Cohort, (final_plot.sel(Type="electrolysis_AEL").values + final_plot.sel(Type="electrolysis_SOEL").values), color = "#FD2724", linestyle ="--", linewidth=lw+0.5, label="MERGED: AEL/SOEL")


ax.set_title(r"H2 market shares - $\mathbf{SSP2\ baseline}$")
# ax.set_title(r"H2 market shares - $\mathbf{SSP2\ VLLO}$")

ax.tick_params(axis="both", labelsize=14)

ax.legend(fontsize=12, loc="upper right")


### CML data

In [ ]:
market_shares_el = market_shares_da.sel(Type=['electrolysis_AEL', 'electrolysis_PEMEL',
       'electrolysis_SOEL'])

fig, ax = plt.subplots(figsize=(10, 6))
for t in market_shares_el["Type"].values:
    market_shares_el.sel(Type=t, Cohort=slice(2010,2070)).plot.line(x="Cohort", ax=ax, label=str(t))
ax.legend()

### Merged

In [ ]:
final_plot = shares_final.sel(Cohort=slice(2005,2070))

fig, ax = plt.subplots(figsize=(10, 6))
lw = 3
ax.plot(final_plot.Cohort, final_plot.sel(Type="electrolysis_PEMEL").values, color="#6F9DFF", linestyle ="--", linewidth=lw+0.5, label="MERGED: PEM")
ax.plot(final_plot.Cohort, final_plot.sel(Type="electrolysis_AEL").values, color = "#BF0502", linestyle ="--", linewidth=lw+0.5, label="MERGED: AEL")
ax.plot(final_plot.Cohort, final_plot.sel(Type="electrolysis_SOEL").values, color = "#FE8583", linestyle ="--", linewidth=lw+0.5, label="MERGED: SOEL")


# ax.set_title(r"H2 market shares - $\mathbf{SSP2\ baseline}$")
ax.set_title(r"H2 market shares - $\mathbf{SSP2\ VLLO}$")

ax.tick_params(axis="both", labelsize=14)

ax.legend(fontsize=12, loc="upper right")




### CML-Merged

In [ ]:

final_plot = shares_final.sel(Cohort=slice(2005,2070))


fig, ax = plt.subplots(figsize=(10, 6))
lw = 3
ax.plot(final_plot.Cohort, final_plot.sel(Type="electrolysis_PEMEL").values, color="#6F9DFF", linestyle ="--", linewidth=lw+0.5, label="MERGED: PEM")
ax.plot(final_plot.Cohort, final_plot.sel(Type="electrolysis_AEL").values, color = "#BF0502", linestyle ="--", linewidth=lw+0.5, label="MERGED: AEL")
ax.plot(final_plot.Cohort, final_plot.sel(Type="electrolysis_SOEL").values, color = "#FE8583", linestyle ="--", linewidth=lw+0.5, label="MERGED: SOEL")

ax.plot(market_shares_da.Cohort, market_shares_da.sel(Type="electrolysis_PEMEL").values, color="#6F9DFF", linestyle ="-", linewidth=lw+0.5, label="CML: PEM")
ax.plot(market_shares_da.Cohort, market_shares_da.sel(Type="electrolysis_AEL").values, color = "#BF0502", linestyle ="-", linewidth=lw+0.5, label="CML: AEL")
ax.plot(market_shares_da.Cohort, market_shares_da.sel(Type="electrolysis_SOEL").values, color = "#FE8583", linestyle ="-", linewidth=lw+0.5, label="CML: SOEL")


ax.set_title(r"H2 market shares - $\mathbf{SSP2\ baseline}$")
# ax.set_title(r"H2 market shares - $\mathbf{SSP2\ VLLO}$")

ax.tick_params(axis="both", labelsize=14)

ax.legend(fontsize=12, loc="upper right")


